# Data Preparation: Market Price Data (DK1)

This notebook handles the ingestion, cleaning, and preprocessing of historical Day-Ahead electricity market price data for the **DK1 (Western Denmark)** bidding zone. 

The objective is to transform an unformatted, variable-resolution raw dataset into a clean, timezone-consistent, **8760-hour** profile suitable for deterministic energy system optimization in PyPSA.

## 🛠️ Data Challenges Addressed

1. **Irregular Headers:** The raw file contains blank rows and metadata in the first three lines that must be bypassed to isolate the numerical arrays.
2. **Variable Temporal Resolution:** The dataset natively switches from **hourly intervals** to **15-minute intervals** mid-year (around October). PyPSA requires a uniform time step across the entire network snapshot horizon.
3. **Timezone Harmonization:** The raw data utilizes local time tracking with varying offsets (+01:00 / +02:00) due to daylight savings. This must be aligned and made timezone-naive to safely couple with meteorological datasets.

## 📋 Methodology Pipeline

* **Ingestion:** Load raw CSV data while skipping the first 3 lines, mapping custom columns (`datetime`, `price`).
* **Timezone Alignment:** Parse strings to UTC to prevent data shifting, convert to Copenhagen local time (`Europe/Copenhagen`), and strip timezone awareness for clean PyPSA snapshot compatibility.
* **Resampling:** Downsample the 15-minute intervals to a uniform 1-hour (`1H`) frequency by calculating the mean price across each hourly block.
* **Reindexing & Interpolation:** Enforce an exact 8760-row index covering the full calendar year. Missing hourly steps (if any) are resolved via linear interpolation.
* **Export:** Save the finalized matrix into `/data/processed/` with a standard `time` index header.

In [1]:
import pandas as pd

# 1. Define paths 
# Since this notebook is inside '00-Data-Preparation', we use '../' to go up one folder 
# into the main directory, then down into the 'data' folder.
raw_file = '../data/raw/electricity_price_2025.csv'
processed_file = '../data/processed/electricity_price_2025_hourly.csv'

# 2. Load the data
# We skip the first 3 messy header rows and manually assign clean column names
df = pd.read_csv(raw_file, skiprows=3, names=['datetime', 'price'])

# 3. Handle Timezones
# Your data has mixed offsets (like +02:00). setting utc=True aligns everything safely.
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
df = df.set_index('datetime')

# Convert from UTC to Danish local time (CET/CEST) to match weather data later
df = df.tz_convert('Europe/Copenhagen')

# PyPSA generally prefers timezone-naive datetimes, so we remove the timezone info
df = df.tz_localize(None)

# 4. Fix the 15-minute interval issue
# We resample the entire dataframe to 1-Hour ('1H') intervals. 
# Using .mean() averages the four 15-minute prices into a single hourly price.
df_hourly = df.resample('1H').mean()

# 5. Enforce an exact 8760-hour year
# We create a perfect hourly index for the year 2025.
# This ensures PyPSA gets exactly 8760 rows, even if the raw data missed a few hours.
full_year = pd.date_range(start='2025-01-01 00:00', end='2025-12-31 23:00', freq='1H')
df_hourly = df_hourly.reindex(full_year)

# If reindexing created any NaN (blank) rows, we fill them using linear interpolation
df_hourly['price'] = df_hourly['price'].interpolate(method='linear')

# 6. Save the processed data
# PyPSA expects the time column to be named 'time'
df_hourly.index.name = 'time'
df_hourly.to_csv(processed_file)

print(f"Data processing complete! Shape: {df_hourly.shape}")
print(df_hourly.head())

Data processing complete! Shape: (8760, 1)
                     price
time                      
2025-01-01 00:00:00    NaN
2025-01-01 01:00:00   1.60
2025-01-01 02:00:00   1.63
2025-01-01 03:00:00   1.09
2025-01-01 04:00:00   0.47


C:\Users\jebat\AppData\Local\Temp\ipykernel_10692\1715903897.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('1H').mean()
C:\Users\jebat\AppData\Local\Temp\ipykernel_10692\1715903897.py:32: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_year = pd.date_range(start='2025-01-01 00:00', end='2025-12-31 23:00', freq='1H')


# Data Preparation: Solar PV Generation

This section processes the solar photovoltaic (PV) generation profiles sourced from **Renewables.ninja**. 

The goal is to extract the raw generation data, strip out the initial metadata rows, and align the timestamps perfectly with our DK1 market price data. Since PyPSA requires strict temporal alignment across all network components, timezone synchronization is the critical step here.

## 🛠️ Processing Steps

1. **Bypass Metadata:** Drop the first 3 rows of system metadata to cleanly read the CSV headers.
2. **Column Isolation:** Keep only the strict UTC `time` column and the `electricity` generation column. The redundant `local_time` column provided by Renewables.ninja is discarded to prevent timezone confusion.
3. **Timezone Synchronization:** Convert the UTC timestamps to Copenhagen local time (`Europe/Copenhagen`) to match the day-ahead market profiles perfectly, then make the index timezone-naive for PyPSA ingestion.

In [2]:
# 1. Define paths 
raw_pv_file = '../data/raw/pv_2025.csv'
processed_pv_file = '../data/processed/pv_2025_hourly.csv'

# 2. Load the data
# The image shows 3 rows of metadata before the headers on row 4.
df_pv = pd.read_csv(raw_pv_file, skiprows=3)

# 3. Filter and Rename Columns
# We only need the UTC 'time' and the 'electricity' columns.
df_pv = df_pv[['time', 'electricity']]
df_pv = df_pv.rename(columns={'electricity': 'pv_generation_kw'})

# 4. Handle Timezones
# The Renewables.ninja 'time' column is strictly UTC.
df_pv['time'] = pd.to_datetime(df_pv['time'], utc=True)
df_pv = df_pv.set_index('time')

# Convert to Danish local time to align perfectly with the market prices
df_pv = df_pv.tz_convert('Europe/Copenhagen')

# Remove timezone info for PyPSA compatibility
df_pv = df_pv.tz_localize(None)

# 5. Save the processed data
df_pv.index.name = 'time'
df_pv.to_csv(processed_pv_file)

print(f"PV Data processing complete! Shape: {df_pv.shape}")
print(df_pv.head())

PV Data processing complete! Shape: (8760, 1)
                     pv_generation_kw
time                                 
2025-01-01 01:00:00               0.0
2025-01-01 02:00:00               0.0
2025-01-01 03:00:00               0.0
2025-01-01 04:00:00               0.0
2025-01-01 05:00:00               0.0


# Data Preparation: Wind Power Generation

This section handles the wind power generation profiles from **Renewables.ninja**. 

Similar to the solar PV data, the raw CSV contains system metadata and uses a UTC timestamp. We will extract the core generation array, synchronize the timezone to Copenhagen local time, and format the data structure to cleanly integrate with the PyPSA network components.

## 🛠️ Processing Steps

1. **Bypass Metadata:** Skip the first 3 rows to read the correct column headers.
2. **Column Isolation:** Extract the absolute UTC `time` column and the `electricity` column, discarding the automatically generated local time to manually enforce our own timezone control.
3. **Timezone Synchronization:** Convert timestamps to `Europe/Copenhagen` to match the DK1 day-ahead market data, then strip the timezone awareness for PyPSA ingestion.

In [3]:
# 1. Define paths 
raw_wind_file = '../data/raw/wind_2025.csv'
processed_wind_file = '../data/processed/wind_2025_hourly.csv'

# 2. Load the data
# Skipping the 3 metadata rows shown in the raw file
df_wind = pd.read_csv(raw_wind_file, skiprows=3)

# 3. Filter and Rename Columns
# Isolating the UTC time and the generation data
df_wind = df_wind[['time', 'electricity']]
df_wind = df_wind.rename(columns={'electricity': 'wind_generation_kw'})

# 4. Handle Timezones
# Convert the string column to a proper UTC datetime object
df_wind['time'] = pd.to_datetime(df_wind['time'], utc=True)
df_wind = df_wind.set_index('time')

# Convert to Danish local time to align with the market prices and PV data
df_wind = df_wind.tz_convert('Europe/Copenhagen')

# Remove timezone info for PyPSA compatibility (timezone-naive)
df_wind = df_wind.tz_localize(None)

# 5. Save the processed data
df_wind.index.name = 'time'
df_wind.to_csv(processed_wind_file)

print(f"Wind Data processing complete! Shape: {df_wind.shape}")
print(df_wind.head())

Wind Data processing complete! Shape: (8760, 1)
                     wind_generation_kw
time                                   
2025-01-01 01:00:00               0.990
2025-01-01 02:00:00               0.988
2025-01-01 03:00:00               0.985
2025-01-01 04:00:00               0.983
2025-01-01 05:00:00               0.981
